[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/spin_groups/spin_groups.ipynb)

# The Spin Groups

This example computes the structure of the spin groups across every signature up to six dimensions. In every signature a rotor is the exponential of a bivector, and the rotors form the signature's spin group, including the rotations of three-dimensional space and the Lorentz transformations of spacetime. Three maps determine how many generators rotate and how many boost, and whether the spinors split into two invariant halves or turn in pairs through a quarter turn that commutes with every rotor.

One nine-dimensional algebra with six positive directions and three negative directions holds every signature up to six dimensions by restriction. From the chosen subspace's types alone, `Bivector.commutator(Bivector.commutator(Bivector)).trace(0, 3)` evaluates the Lie algebra's invariant form in exact integers as `2 * (n - 2)` times `Bivector | Bivector`, `positives >> Bivector` separates rotating planes from boosting planes by sign, and `pseudoscalar * Even` gives the centre's action on spinors. In four Euclidean dimensions the same pseudoscalar splits the Lie algebra into two commuting isoclinic halves `(1 + xyzw) / 2` and `(1 - xyzw) / 2` whose flows trace the Hopf fibration and its mirror image on the three-sphere.

In matrix Lie-group notation, realizing $\mathrm{Spin}(p, q)$ as a double cover of $\mathrm{SO}(p, q)$ switches matrix rings signature by signature across $\mathrm{SU}(2)$, $\mathrm{SL}(2, \mathbb{R})$, $\mathrm{SL}(2, \mathbb{C})$, $\mathrm{Sp}(1, 1)$, and $\mathrm{SU}(2, 2)$, builds $\mathfrak{so}(p, q)$ from metric-indefinite skew matrices $X^\top \eta + \eta X = 0$ with Killing form $\operatorname{tr}(\operatorname{ad}_X \operatorname{ad}_Y)$, and writes the Cartan involution as $-X^\top$ alongside separate chiral projectors $\tfrac{1}{2}(I \pm \gamma_5)$ in a chosen Dirac-matrix representation.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import Algebra, NumpyContext
from numga.extensor import Extensor
from numga.gatype import GAType
from examples.util.animation import save_figure, show_states
from examples.math.spin_groups import render, scenarios

np.set_printoptions(precision=4, suppress=True)

# Six directions squaring to plus one and three squaring to minus one.
ga = Algebra("x+y+z+w+v+u+t-s-r-")
context = NumpyContext(ga)
mv = context.multivector
positive_names, negative_names = "xyzwvu", "tsr"

# Points of the four Euclidean directions x y z w are the antivectors containing the five remaining directions.
Euclidean = ga.gatype.vector().derive.within(Algebra("x+y+z+w+")).derive.dual   # Antivector-
Antitrivector = ga.gatype.antitrivector()                                       # Antitrivector-

# The four-dimensional Euclidean pseudoscalar commutes with bivectors and squares to one, splitting each
# generator into isoclinic halves that turn every plane along with or against its orthogonal complement.
euclidean_pseudoscalar = mv.xyzw                                                # [] Quadvector+
plus = (1 + euclidean_pseudoscalar) / 2                                         # [] Even+
minus = (1 - euclidean_pseudoscalar) / 2                                        # [] Even+


def signature(p: int, q: int) -> tuple[GAType, GAType, Extensor, Extensor]:
    """Choosing the first `p` positive and `q` negative directions yields their `Bivector` and `Even`
    types alongside the product of the positive directions and the signature's pseudoscalar."""
    names = positive_names[:p] + negative_names[:q]
    directions = Algebra.from_pqr(p, q, 0, names)
    Bivector = ga.gatype.bivector().derive.within(directions)
    Even = ga.gatype.even().derive.within(directions)
    positives = getattr(mv, positive_names[:p])
    pseudoscalar = getattr(mv, names)
    return Bivector, Even, positives, pseudoscalar


# Three positive directions x y z and one negative direction t pick out spacetime signature (3, 1).
Bivector, Even, positives, pseudoscalar = signature(3, 1)
print(Bivector.output_subspace)
print(Even.output_subspace)

## Invariant form and inner product

Bivectors generate the rotors and combine under the commutator. Tracing what the double commutator `Bivector.commutator(Bivector.commutator(Bivector))` of two generators does to a third defines the Lie algebra's invariant bilinear form on pairs of bivectors. Built from types alone, the form has exact integer entries and equals `2 * (n - 2)` times the algebra's own inner product `Bivector | Bivector`, which equals `4 * (Bivector | Bivector)` in four dimensions.

In Lie-algebra notation, this bilinear form reads as the Killing form $K(X, Y) = \operatorname{tr}(\operatorname{ad}_X \operatorname{ad}_Y)$.

In [ ]:
def invariant_form(Bivector: GAType) -> Extensor:
    """The trace of the double commutator on `Bivector` defines the Lie algebra's invariant bilinear
    form (`[] Scalar+ <- (Bivector+, Bivector+)`)."""
    return Bivector.commutator(Bivector.commutator(Bivector)).trace(0, 3)       # [] Scalar+ <- (Bivector+, Bivector+)


planes = mv(Bivector, np.eye(6))                                                # [6] Bivector+
form = invariant_form(Bivector)                                                 # [] Scalar+ <- (Bivector+, Bivector+)
inner = Bivector | Bivector                                                     # [] Scalar+ <- (Bivector+, Bivector+)
form_entries = form.materialize(context)(planes[:, None], planes[None, :])      # [6, 6] Scalar+
inner_entries = inner.materialize(context)(planes[:, None], planes[None, :])    # [6, 6] Scalar+
print(form_entries.to_array())

## Rotations and boosts

Sandwiching the bivectors with the product of the chosen positive directions, `positives >> Bivector`, defines an involution whose square is the identity. Planes within the positive or negative directions are fixed, square to minus one, and generate rotations; planes mixing a positive with a negative direction are negated, square to plus one, and generate boosts. In spacetime signature `(3, 1)`, the involution fixes the three spatial planes `xy`, `yz`, `zx` and negates the three boost planes containing `t`.

In Lie-algebra notation, this involution reads as the Cartan involution splitting $\mathfrak{so}(p, q) = \mathfrak{k} \oplus \mathfrak{p}$ into its maximal compact subalgebra $\mathfrak{k}$ and boost subspace $\mathfrak{p}$.

In [ ]:
involution = positives >> Bivector                                              # [] Bivector+ <- Bivector+
signs = involution.eigvals()                                                    # [6] Scalar+
# One spatial plane of x y z and one boost plane containing t.
xy_image, xt_image = involution(mv.xy), involution(mv.xt)                       # [] Bivector+, Bivector+
print("eigenvalues:", signs.real().to_array())
print("xy ->\n", xy_image.coefficient_table())
print("xt ->\n", xt_image.coefficient_table())

## Pseudoscalar action on spinors

In even dimensions the pseudoscalar of the chosen directions is itself an even multivector, so `pseudoscalar * Even` maps spinors to spinors, commutes with every rotor, and squares to `pseudoscalar.squared()`. When that square is plus one, as in signature `(4, 0)`, its eigenvalues are plus and minus one and `(1 + pseudoscalar) / 2` and `(1 - pseudoscalar) / 2` project the spinors onto two invariant chiral halves. When that square is minus one, as in signature `(3, 1)`, multiplying by the pseudoscalar squares to minus the identity on spinors, has no real eigenvalues, and turns every pair of spinor components through one shared quarter turn.

In matrix spinor notation, signature $(3, 1)$ spinors are written as two-component complex Weyl spinors acted on by $\mathrm{SL}(2, \mathbb{C})$ where multiplication by $i$ gives that quarter turn, whereas signature $(4, 0)$ spinors split into two independent two-component spinors acted on by the two factors of $\mathrm{SU}(2) \times \mathrm{SU}(2)$.

In [ ]:
lorentzian_action = (pseudoscalar * Even).eigvals()                             # [8] Scalar+
Bivector_4d, EuclideanEven, _, _ = signature(4, 0)
euclidean_action = (euclidean_pseudoscalar * EuclideanEven).eigvals()           # [8] Scalar+
print("signature (3, 1), pseudoscalar squared =", pseudoscalar.squared().to_array(), "  eigenvalues:", lorentzian_action.to_array())
print("signature (4, 0), pseudoscalar squared =", euclidean_pseudoscalar.squared().to_array(), "  eigenvalues:", euclidean_action.real().to_array())

## Isoclinic split and Hopf flows in four dimensions

In four Euclidean dimensions the pseudoscalar maps bivectors to bivectors, so its chiral projectors `plus = (1 + xyzw) / 2` and `minus = (1 - xyzw) / 2` split the Lie algebra itself into two commuting three-dimensional subalgebras. Projecting any generator onto `plus` and `minus` gives two commuting bivectors whose exponentials multiply to the full rotor, and each factor turns every plane through one shared angle.

Following a single isoclinic factor carries every point of the three-sphere around a great circle, and stereographic projection maps those circles to linked circles in three-dimensional space that form the Hopf fibration for `plus` and its mirror image for `minus`. Combining the two factors at commensurable rates traces torus knots on their shared invariant tori, where the generator `2 * xy + 3 * zw` decomposes into `5 * against - along` and closes after one period as a trefoil knot.

In quaternion notation, identifying $\mathbb{R}^4$ with $\mathbb{H}$ writes a four-dimensional rotation as the two-sided action $v \mapsto q_L\, v\, \bar{q}_R$ of a pair of unit quaternions $(q_L, q_R) \in \mathrm{Sp}(1) \times \mathrm{Sp}(1)$.

In [ ]:
def isoclinic_factors(generator: Extensor) -> tuple[Extensor, Extensor]:
    """Projecting a four-dimensional bivector `generator` onto `plus` and `minus` and exponentiating
    gives its two commuting isoclinic rotor factors (`[] Rotor+, Rotor+`)."""
    left_factor = (plus * generator).exp()                                      # [] Rotor+
    right_factor = (minus * generator).exp()                                    # [] Rotor+
    return left_factor, right_factor


# A four-dimensional generator turning across xy, zw, and xz simultaneously.
generator = 0.4 * mv.xy + 0.9 * mv.zw - 0.3 * mv.xz                             # [] Bivector+
rotor = generator.exp()                                                         # [] Rotor+
left_factor, right_factor = isoclinic_factors(generator)                        # [] Rotor+, Rotor+
left_angles = (left_factor >> Euclidean).eigvals().real().arccos()              # [4] Scalar+
right_angles = (right_factor >> Euclidean).eigvals().real().arccos()            # [4] Scalar+
print("angles of left factor: ", left_angles.to_array())
print("angles of right factor:", right_angles.to_array())

In [ ]:
n_azimuths = scenarios.n_azimuths
heights = scenarios.heights
n_steps = scenarios.n_steps
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def stereographic(points: Euclidean) -> Euclidean:
    """Stereographic projection from the pole at `-w` maps `points` (`[...] Antivector-`) on the
    three-sphere into the three-dimensional space of `x`, `y`, and `z` (`[...] Antivector-`)."""
    w_coordinate = mv.w & points                                                # [...] Scalar+
    return (points - mv.w.dual() * w_coordinate) / (1 + w_coordinate)           # [...] Antivector-


def isoclinic_orbits(n_azimuths: int, heights: np.ndarray, n_steps: int
                     ) -> tuple[Euclidean, Euclidean, Euclidean]:
    """Stereographic projections of the two opposite isoclinic circle families (`[n_heights,
    n_azimuths, n_steps + 1] Antivector-`) and of the `(2, 3)` torus-knot orbits (`[n_heights, 1,
    n_steps + 1] Antivector-`) on the three-sphere."""
    n_heights = len(heights)
    # Turning xy along with its orthogonal plane zw and against it gives the two isoclinic generators.
    along, against = plus * mv.xy, minus * mv.xy                                # [] Bivector+, Bivector+
    combined = 2 * mv.xy + 3 * mv.zw                                            # [] Bivector+
    # Starting points turn from +x toward +z by each height angle and around the xy plane by each azimuth.
    azimuths = np.linspace(0, 2 * np.pi, n_azimuths, endpoint=False)
    azimuth_turns = (mv.xy * (-azimuths / 2)).exp()                             # [n_azimuths] Rotor+
    height_turns = (mv.xz * (-heights[:, None] / 2)).exp()                      # [n_heights, 1] Rotor+
    starts = (azimuth_turns * height_turns) >> mv.x.dual()                      # [n_heights, n_azimuths] Antivector-
    # Because the sandwich product turns planes by twice the rotor angle, pi completes one full turn.
    turn_angles = np.linspace(0, np.pi, n_steps + 1)
    along_rotors = (2 * along * turn_angles).exp()                              # [n_steps + 1] Rotor+
    against_rotors = (2 * against * turn_angles).exp()                          # [n_steps + 1] Rotor+
    combined_rotors = (combined * turn_angles).exp()                            # [n_steps + 1] Rotor+
    left_orbits = stereographic(along_rotors >> starts[..., None])              # [n_heights, n_azimuths, n_steps + 1] Antivector-
    right_orbits = stereographic(against_rotors >> starts[..., None])           # [n_heights, n_azimuths, n_steps + 1] Antivector-
    knotted_orbits = stereographic(combined_rotors >> starts[:, :1, None])      # [n_heights, 1, n_steps + 1] Antivector-
    return left_orbits, right_orbits, knotted_orbits


left_orbits, right_orbits, knotted_orbits = isoclinic_orbits(n_azimuths, heights, n_steps)
save_figure(render.draw_flows(left_orbits, right_orbits, knotted_orbits), "spin_groups_flows");

The left and middle panels show the stereographically projected circle families of the two opposite isoclinic flows across three nested Clifford tori, where hue tracks the starting azimuth around the `xy` plane and brightness tracks the torus height angle toward `z`. Every circle in the left panel links every other circle in its family with one handedness, the middle panel twists the circles with the mirror-image handedness, and the right panel shows one `(2, 3)` trefoil knot on each torus from combining both flows.

Joining the separation between two loop segments with both segment displacements gives a signed `Antitrivector` volume element, so Gauss's linking integral evaluates directly to plus or minus one unit three-volume and reverses sign under reflection.

In [ ]:
def linking(first: Euclidean, second: Euclidean) -> Antitrivector:
    """Gauss's linking integral of two closed polygonal loops (`[n_vertices] Antivector-`) in
    three-dimensional space yields an integer multiple of the unit volume (`[] Antitrivector-`)
    whose sign reverses under reflection."""
    n_segments = first.shape[0] - 1
    first_steps = first[1:] - first[:-1]                                        # [n_segments] Antivector-
    second_steps = second[1:] - second[:-1]                                     # [n_segments] Antivector-
    first_midpoints = 0.5 * (first[1:] + first[:-1])                            # [n_segments] Antivector-
    second_midpoints = 0.5 * (second[1:] + second[:-1])                         # [n_segments] Antivector-
    separations = first_midpoints[:, None] - second_midpoints[None, :]          # [n_segments, n_segments] Antivector-
    # Joining the separation vector with both segment displacements measures the signed spanned volume.
    volumes = separations & first_steps[:, None] & second_steps[None, :]        # [n_segments, n_segments] Antitrivector-
    lengths = separations.antinorm()                                            # [n_segments, n_segments] Scalar+
    return (volumes / lengths ** 3).sum() / (4 * np.pi)                         # [] Antitrivector-


linking_along = linking(left_orbits[0, 0], left_orbits[-1, n_azimuths // 3])     # [] Antitrivector-
linking_against = linking(right_orbits[0, 0], right_orbits[-1, n_azimuths // 3])  # [] Antitrivector-
print("linking of two orbits of the left flow:\n", linking_along.coefficient_table())
print("linking of two orbits of the right flow:\n", linking_against.coefficient_table())
shown_steps = show_states(
    scenarios.flow_steps(n_steps, n_frames),
    render.animate_flows,
    left_orbits,
    right_orbits,
    knotted_orbits,
    name="spin_groups_isoclinic",
    duration_ms=frame_duration_ms,
    n_frames=n_frames,
)

Points in the left and middle panels advance at uniform speed around their linked Hopf circles in opposite senses while evenly spaced riders trace each trefoil knot in the right panel, returning every family to its starting configuration after one period.

## Every signature up to six dimensions

Evaluating the same three maps across every signature `(p, q)` from three to six dimensions classifies the spin groups in one pass. The ratio of `invariant_form(Bivector)` to `Bivector | Bivector` equals `2 * (p + q - 2)` in every signature, `positives >> Bivector` fixes the `p * (p - 1) // 2 + q * (q - 1) // 2` rotation planes and negates the `p * q` boost planes, and in even dimensions `pseudoscalar * Even` either splits the spinors into two invariant chiral halves when `pseudoscalar.squared()` is `+1` or turns spinor pairs through a quarter turn when `pseudoscalar.squared()` is `-1`. In odd dimensions the pseudoscalar is an odd multivector outside `Even`, so the spinors stay irreducible.

In matrix Lie-group notation, these twelve spin groups are written separately via their low-dimensional matrix isomorphisms as $\mathrm{Spin}(3, 0) \cong \mathrm{SU}(2)$, $\mathrm{Spin}(2, 1) \cong \mathrm{SL}(2, \mathbb{R})$, $\mathrm{Spin}(4, 0) \cong \mathrm{SU}(2) \times \mathrm{SU}(2)$, $\mathrm{Spin}(3, 1) \cong \mathrm{SL}(2, \mathbb{C})$, $\mathrm{Spin}(2, 2) \cong \mathrm{SL}(2, \mathbb{R}) \times \mathrm{SL}(2, \mathbb{R})$, $\mathrm{Spin}(5, 0) \cong \mathrm{Sp}(2)$, $\mathrm{Spin}(4, 1) \cong \mathrm{Sp}(1, 1)$, $\mathrm{Spin}(3, 2) \cong \mathrm{Sp}(4, \mathbb{R})$, $\mathrm{Spin}(6, 0) \cong \mathrm{SU}(4)$, $\mathrm{Spin}(5, 1) \cong \mathrm{SL}(2, \mathbb{H})$, $\mathrm{Spin}(4, 2) \cong \mathrm{SU}(2, 2)$, and $\mathrm{Spin}(3, 3) \cong \mathrm{SL}(4, \mathbb{R})$.

In [ ]:
signatures = scenarios.signatures
seed = scenarios.seed
even_signatures = tuple(pair for pair in signatures if sum(pair) % 2 == 0)

In [ ]:
def signature_rows(signatures: tuple[tuple[int, int], ...], seed: int) -> Iterator[tuple]:
    """Yields `(p, q, form, ratio, signs, square)` across `signatures`: the exact invariant form,
    its ratio to the inner product on a random bivector, the involution's eigenvalues, and the
    pseudoscalar's square."""
    rng = np.random.default_rng(seed)
    for p, q in signatures:
        Bivector_pq, Even_pq, positives_pq, pseudoscalar_pq = signature(p, q)
        n_planes = len(Bivector_pq.output_subspace.masks)
        form_pq = invariant_form(Bivector_pq)                                   # [] Scalar+ <- (Bivector+, Bivector+)
        sample = mv(Bivector_pq, rng.normal(size=n_planes))                     # [] Bivector+
        ratio = form_pq.materialize(context)(sample, sample) / (sample | sample)  # [] Scalar+
        involution_pq = positives_pq >> Bivector_pq                             # [] Bivector+ <- Bivector+
        signs_pq = involution_pq.eigvals()                                      # [n_planes] Scalar+
        yield p, q, form_pq, ratio, signs_pq, pseudoscalar_pq.squared()


def spinor_centres(signatures: tuple[tuple[int, int], ...]) -> Iterator[tuple]:
    """Yields `(p, q, action)` across even-dimensional `signatures`: the eigenvalues of the
    pseudoscalar acting on the spinors (`[n_spinors] Scalar+`)."""
    for p, q in signatures:
        _, Even_pq, _, pseudoscalar_pq = signature(p, q)
        n_spinors = len(Even_pq.output_subspace.masks)
        action = (pseudoscalar_pq * Even_pq).eigvals()                          # [n_spinors] Scalar+
        yield p, q, action


rows = list(signature_rows(signatures, seed))
actions = list(spinor_centres(even_signatures))
for p, q, _, ratio, row_signs, square in rows:
    print(
        (p, q),
        "form / inner:", ratio.to_array(),
        " involution eigenvalues:", row_signs.real().to_array(),
        " pseudoscalar squared:", square.to_array(),
    )
for p, q, action in actions:
    print((p, q), "pseudoscalar eigenvalues on spinors:", action.to_array())

In [ ]:
# checks
# In signature (3, 1) the invariant form equals four times the inner product, the involution fixes
# spatial rotation planes and negates boost planes, and the pseudoscalar squares to -1 on spinors
# while squaring to +1 in signature (4, 0).
assert ((form_entries - 4 * inner_entries).abs() <= 0).all()
assert (xy_image - mv.xy).scalar_norm_squared() < 1e-12
assert (xt_image + mv.xt).scalar_norm_squared() < 1e-12
assert (signs.real() > 0).sum() == 3
assert (signs.real() < 0).sum() == 3
assert ((lorentzian_action.squared() + 1).abs() < 1e-12).all()
assert ((euclidean_action.squared() - 1).abs() < 1e-12).all()
# The four-dimensional rotor factors into two commuting isoclinic halves, each turning every plane by one angle,
# while closed Hopf fibres link with unit volume of opposite sign and 2 * xy + 3 * zw equals 5 * against - along.
assert (left_factor * right_factor - rotor).scalar_norm_squared() < 1e-12
assert (left_factor * right_factor - right_factor * left_factor).scalar_norm_squared() < 1e-12
assert ((left_angles - left_angles[0]).abs() < 1e-9).all()
assert ((right_angles - right_angles[0]).abs() < 1e-9).all()
along, against = plus * mv.xy, minus * mv.xy
assert (2 * mv.xy + 3 * mv.zw - (5 * against - along)).scalar_norm_squared() < 1e-12
assert ((left_orbits[..., 0] - left_orbits[..., -1]).antinorm() < 1e-5).all()
assert ((knotted_orbits[..., 0] - knotted_orbits[..., -1]).antinorm() < 1e-5).all()
assert ((mv.xyz & linking_along).abs() - 1).abs() < 1e-2
assert (linking_along + linking_against).antinorm() < 1e-2
assert len(shown_steps) == n_frames
# Across every signature up to six dimensions, the invariant form equals 2 * (p + q - 2) times the inner product,
# the involution counts rotation and boost planes, and the even-dimensional pseudoscalar eigenvalues match its square.
for p, q, row_form, _, row_signs, _ in rows:
    Bivector_pq, _, _, _ = signature(p, q)
    basis_planes = mv(Bivector_pq, np.eye(len(Bivector_pq.output_subspace.masks)))
    row_difference = (row_form - 2 * (p + q - 2) * (Bivector_pq | Bivector_pq)).materialize(context)(
        basis_planes[:, None], basis_planes[None, :],
    )
    assert (row_difference.abs() <= 0).all()
    real_signs = row_signs.real()
    assert (real_signs > 0).sum() == p * (p - 1) // 2 + q * (q - 1) // 2
    assert (real_signs < 0).sum() == p * q
squares = {(p, q): square for p, q, _, _, _, square in rows}
for p, q, action in actions:
    assert ((action.squared() - squares[p, q]).abs() < 1e-12).all()